In [1]:
# ================================================================
#  NB-CROSS — CROSS-DATASET VALIDATION  (Appidi Kannada-English tweets)
#
#  Two experiments, kept separate:
#
#    Q1 REPLICATION  train and test WITHIN the external corpus.
#                    If the high-recovery normalizers damage performance there
#                    too, the finding is a property of the normalizers rather
#                    than of our corpus. This is the result that matters.
#
#    Q2 TRANSFER     train on our corpus, test on the external one.
#                    Weaker and confounded by domain shift, reported because
#                    reviewers expect it.
#
#  pipeline.py is copied from the NB1 output, so the thirteen methods are
#  byte-identical to the main experiments. Nothing is re-implemented.
#
#  Add Input:
#    final-emotion-dataset   Gold_dataset + Noisy + Synset + Synonyms
#    NB1 output              pipeline.py and splits/
#    appidi corpus           appidi_4class.csv
#    xlit_cache.csv          optional; without it only the itrans method runs
#
#  CPU is sufficient. Runtime is roughly 15 minutes.
# ================================================================
import subprocess, sys, os, glob, shutil
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library aksharamukha '
   'indic-transliteration scikit-learn scipy')

for p in ("pipeline.py", "/kaggle/working/pipeline.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)

src = glob.glob("/kaggle/input/**/pipeline.py", recursive=True)
assert src, "attach the NB1 notebook output; pipeline.py is required"
shutil.copy(src[0], "/kaggle/working/pipeline.py")
print("\npipeline from:", src[0])

print("\nCSV files visible:")
for f in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
    try: n = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
    except Exception: n = "?"
    print(f"  {n:>8} rows  {f}")


+ pip install -q regex emoji wordfreq indic-nlp-library aksharamukha indic-transliteration scikit-learn scipy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ================================================================
#  CELL 1 — load the external corpus
#
#  The Appidi corpus already uses the four categories, so no label mapping is
#  needed and no rows are dropped. If you substitute another corpus, set the
#  three variables below and extend LABEL_MAP. Unmapped values are DROPPED and
#  counted, never merged onto an existing class.
# ================================================================
import pandas as pd, numpy as np, glob, os, sys
sys.path.insert(0, "/kaggle/working")
import pipeline
from pipeline import *
FP = pipeline.fingerprint()

LABELS = ["happy", "sad", "anger", "disgust"]

# ---- SET THESE ----------------------------------------------------------
EXT_NAME  = "appidi_4class.csv"     # filename to look for
EXT_TEXT  = "tweet"
EXT_LABEL = "label"
LABEL_MAP = {"happy": "happy", "sad": "sad",
             "anger": "anger", "disgust": "disgust"}
# -------------------------------------------------------------------------

hits = [f for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if os.path.basename(f).lower() == EXT_NAME.lower()]
assert hits, (f"{EXT_NAME} not found. Attach it as a dataset, or change "
              f"EXT_NAME. CSVs visible are listed in CELL 0.")
EXT_PATH = hits[0]
print("external corpus:", EXT_PATH)

ext = pd.read_csv(EXT_PATH, dtype=str, keep_default_na=False)
ext.columns = [c.strip().lower().lstrip("\ufeff") for c in ext.columns]
assert EXT_TEXT in ext.columns and EXT_LABEL in ext.columns, \
    f"expected columns '{EXT_TEXT}' and '{EXT_LABEL}', found {list(ext.columns)}"
print(f"loaded {len(ext):,} rows | columns: {list(ext.columns)}")

raw_labels = ext[EXT_LABEL].str.strip().str.lower()
print("\nlabel values:")
for k, v in raw_labels.value_counts().items():
    t = LABEL_MAP.get(k)
    print(f"   {str(k):<14}{v:>7,}   -> {t if t else 'UNMAPPED, dropped'}")

# case folded once, exactly as load_corpus does for the main corpus
ext["comment"] = ext[EXT_TEXT].str.replace(r"\s+", " ", regex=True).str.strip().str.lower()
ext["label"] = raw_labels.map(LABEL_MAP)
dropped = int(ext.label.isna().sum())
ext = ext.dropna(subset=["label"])
ext = ext[ext.comment.str.strip() != ""].drop_duplicates(subset=["comment"]).reset_index(drop=True)
ext["id"] = ["x%06d" % i for i in range(len(ext))]

print(f"\nusable rows: {len(ext):,}   dropped: {dropped:,}")
print("class distribution:", ext.label.value_counts().to_dict())
assert len(ext) >= 500 and ext.label.nunique() == 4, \
    "too few rows or fewer than four classes present"

# how the external corpus differs from ours, for the limitations paragraph
# pandas .str.contains only accepts patterns from the standard re module, so
# the script test is applied with .apply rather than a compiled regex pattern
def _has_kn(t): return any('\u0c80' <= c <= '\u0cff' for c in str(t))
print(f"\nmean length: {ext.comment.str.len().mean():.1f} characters")
print(f"contains Kannada script: {100*ext.comment.apply(_has_kn).mean():.1f}% of comments")
print("  (main corpus: 20.7% of TOKENS are in Kannada script)")
print("Class balance is uneven, so macro-F1 is the appropriate metric.")


pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True
external corpus: /kaggle/input/datasets/tejaswinina/cross-dataset-appidi/appidi_4class.csv
loaded 4,413 rows | columns: ['tweet', 'label']

label values:
   anger           1,395   -> anger
   happy           1,257   -> happy
   disgust           950   -> disgust
   sad               811   -> sad

usable rows: 4,394   dropped: 0
class distribution: {'anger': 1391, 'happy': 1251, 'disgust': 947, 'sad': 805}

mean length: 170.6 characters
contains Kannada script: 0.4% of comments
  (main corpus: 20.7% of TOKENS are in Kannada script)
Class balance is uneven, so macro-F1 is the appropriate metric.


In [3]:
# ================================================================
#  CELL 2 — build the thirteen methods on the external corpus
#
#  Label evidence and the phonetic bucket elections are rebuilt on the
#  EXTERNAL training split. Reusing our corpus's statistics would leak.
# ================================================================
from sklearn.model_selection import train_test_split
from collections import Counter

norm_map, F2E, LEX = build_keel()

etr, ete = train_test_split(ext, test_size=0.25, stratify=ext.label,
                            random_state=SEED)
etr = etr.reset_index(drop=True); ete = ete.reset_index(drop=True)
print(f"external train {len(etr):,} | test {len(ete):,}")
print("test distribution:", ete.label.value_counts().to_dict())

for d in (ext, etr, ete):
    d["text_raw"] = d.comment.map(lambda t: " ".join(process_comment(t)[0]))

evidence = LabelEvidence(etr, "comment", "label")
TRAIN_FREQ = Counter()
for t in etr.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"external train vocab {len(TRAIN_FREQ):,} | evidence over "
      f"{len(evidence.counts):,} forms")

try:
    from indicnlp import common
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    for c in glob.glob("/kaggle/input/**/indic_nlp_resources*", recursive=True):
        common.set_resources_path(c); break
    pipeline._indic_norm = IndicNormalizerFactory().get_normalizer("kn")
    _ = pipeline._indic_norm.normalize("ಪರೀಕ್ಷೆ"); print("IndicNLP: FULL")
except Exception as e:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)", str(e)[:60])

# the cache was built on our corpus, so coverage here will be partial;
# uncovered forms pass through unchanged and the rate is reported
xlit = load_xlit(ext, "comment", build_live=False)

WANTED = ["raw", "keel", "kope", "metaphone", "terun",
          "soundex", "soundex_guard", "indicsoundex", "indicsoundex_guard",
          "lccn_base", "lccn", "lccn_itrans", "lccn_indicxlit"]
avail = set(xlit)
ARMS = [a for a in WANTED
        if not ("itrans" in a or "indicxlit" in a)
        or ("indicxlit" in a and "indicxlit" in avail)
        or ("itrans" in a and "indicxlit" not in a and "itrans" in avail)]
print(f"\nmethods ({len(ARMS)}):", ARMS)

A, LCCN_OBJ, veto = build_arms(norm_map, F2E, evidence, xlit, ARMS, LEX=LEX)
for d in (etr, ete):
    for a in A: d[f"text_{a}"] = d.comment.map(A[a])
print("methods applied")

bad = 0
for s in ext.comment.sample(min(200, len(ext)), random_state=SEED):
    ref = len(A["raw"](s).split())
    for n, f in A.items():
        if len(f(s).split()) != ref: bad += 1
print("all methods consistent" if bad == 0 else f"{bad} token-count mismatches")

for n, o in LCCN_OBJ.items(): o.report(n)
LEX.report()

rows = []
for a in ARMS:
    if a == "raw": continue
    c = (etr[f"text_{a}"] != etr.text_raw).mean() * 100
    ch = tot = 0
    for x, y in zip(etr.text_raw, etr[f"text_{a}"]):
        tx, ty = x.split(), y.split(); tot += len(tx)
        ch += sum(1 for p, q in zip(tx, ty) if p != q)
    rows.append({"method": a, "comment_change_%": round(c, 2),
                 "token_change_%": round(100*ch/max(tot, 1), 2)})
cr_ext = pd.DataFrame(rows)
print("\n=== CHANGE RATE ON THE EXTERNAL CORPUS ===")
print(cr_ext.to_string(index=False))
cr_ext.to_csv("/kaggle/working/cross_change_rates.csv", index=False)


  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
external train 3,295 | test 1,099
test distribution: {'anger': 348, 'happy': 313, 'disgust': 237, 'sad': 201}
external train vocab 21,272 | evidence over 21,272 forms
IndicNLP: FULL
transliteration backend: ITRANS (indic_transliteration.sanscript)
converted 17,989 of 17,996 romanized types to Kannada script (100.0%)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
indicxlit arms SKIPPED: no cache attached and the model cannot
be imported here (needs the Python 3.10 image).
Wrote 17,996 romanized types to /kaggle/working/romanized_vocab.csv
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

transliteration backends ready: ['itrans']

methods (12): ['raw', 'keel', 

In [4]:
# ================================================================
#  CELL 3 — Q1 REPLICATION: train and test within the external corpus
# ================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

MODELS = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=2000, C=4.0,
                                                      class_weight="balanced"),
    "SVM":                 lambda: LinearSVC(C=0.5, class_weight="balanced"),
    "Naive Bayes":         lambda: MultinomialNB(alpha=0.3),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=400,
                                                          n_jobs=-1, random_state=SEED),
}
def vec():
    return make_union(
        TfidfVectorizer(analyzer="word",    ngram_range=(1,2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=3, sublinear_tf=True))

rows = []
for name, ctor in MODELS.items():
    r = {"model": name}
    for a in ARMS:
        v = vec()
        Xtr = v.fit_transform(etr[f"text_{a}"]); Xte = v.transform(ete[f"text_{a}"])
        r[a] = round(f1_score(ete.label, ctor().fit(Xtr, etr.label).predict(Xte),
                              average="macro"), 4)
    rows.append(r)
    best = max(ARMS, key=lambda a: r[a])
    print(f"  {name:<22} raw {r['raw']:.4f}   best {best} ({r[best]:.4f})")
rep = pd.DataFrame(rows)
rep.to_csv("/kaggle/working/cross_replication.csv", index=False)

print("\n=== Q1 REPLICATION, macro-F1 on the external corpus ===")
print(rep.to_string(index=False))
d = rep.copy()
for a in ARMS:
    if a != "raw": d[a] = (rep[a] - rep["raw"]).round(4)
print("\ndelta vs raw:")
print(d[["model"] + [a for a in ARMS if a != "raw"]].to_string(index=False))

print("\nper-class report, raw baseline, logistic regression:")
v = vec(); Xtr = v.fit_transform(etr.text_raw); Xte = v.transform(ete.text_raw)
p = LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced").fit(Xtr, etr.label).predict(Xte)
print(classification_report(ete.label, p, digits=4, zero_division=0))


  Logistic Regression    raw 0.4950   best lccn_base (0.5166)
  SVM                    raw 0.4860   best lccn_base (0.4945)
  Naive Bayes            raw 0.4182   best terun (0.4409)
  Random Forest          raw 0.4232   best lccn (0.4243)

=== Q1 REPLICATION, macro-F1 on the external corpus ===
              model    raw   keel   kope  metaphone  terun  soundex  soundex_guard  indicsoundex  indicsoundex_guard  lccn_base   lccn  lccn_itrans
Logistic Regression 0.4950 0.5000 0.4923     0.4783 0.4966   0.4604         0.4598        0.4547              0.4646     0.5166 0.5052       0.4970
                SVM 0.4860 0.4887 0.4855     0.4622 0.4811   0.4613         0.4567        0.4509              0.4374     0.4945 0.4927       0.4875
        Naive Bayes 0.4182 0.4247 0.4154     0.4191 0.4409   0.4002         0.4040        0.4030              0.4102     0.4290 0.4189       0.4028
      Random Forest 0.4232 0.4175 0.4130     0.4202 0.4100   0.4150         0.3937        0.4061              0.

In [5]:
# ================================================================
#  CELL 4 — does the ORDERING replicate?
#
#  Absolute scores are not comparable across corpora of different size and
#  platform, so the test is whether the RANKING of methods survives. This is
#  the result the paper reports.
# ================================================================
from scipy.stats import spearmanr, pearsonr

# main-corpus means across the nine classifiers
MAIN = {"raw":0.7797, "keel":0.7776, "kope":0.7807, "metaphone":0.7649,
        "terun":0.7739, "soundex":0.7536, "soundex_guard":0.7578,
        "indicsoundex":0.7530, "indicsoundex_guard":0.7594,
        "lccn_base":0.7783, "lccn":0.7776, "lccn_itrans":0.7795,
        "lccn_indicxlit":0.7828}

ext_mean = {a: rep[a].mean() for a in ARMS if a in rep.columns}
shared = [a for a in ext_mean if a in MAIN]
x = [MAIN[a] for a in shared]; y = [ext_mean[a] for a in shared]

rho, p_rho = spearmanr(x, y); r, p_r = pearsonr(x, y)
print(f"methods compared: {len(shared)}")
print(f"Spearman rank correlation between corpora: rho = {rho:+.3f}  p = {p_rho:.4f}")
print(f"Pearson: r = {r:+.3f}  p = {p_r:.4f}\n")

cmp = pd.DataFrame({"method": shared,
                    "main (9 models)": [round(MAIN[a], 4) for a in shared],
                    "external (4 models)": [round(ext_mean[a], 4) for a in shared]})
cmp["main rank"] = cmp["main (9 models)"].rank(ascending=False).astype(int)
cmp["ext rank"]  = cmp["external (4 models)"].rank(ascending=False).astype(int)
cmp["shift"] = cmp["ext rank"] - cmp["main rank"]
cmp = cmp.sort_values("main rank")
print(cmp.to_string(index=False))
cmp.to_csv("/kaggle/working/cross_ranking.csv", index=False)

# recovery against performance, merging methods only, as in the main paper
rec = dict(zip(cr_ext.method, cr_ext["token_change_%"])); rec["raw"] = 0.0
merging = [a for a in shared if a != "raw" and "itrans" not in a and "indicxlit" not in a]
if len(merging) >= 4:
    rr = [rec[a] for a in merging]; ff = [ext_mean[a] for a in merging]
    r2, p2 = pearsonr(rr, ff); s2, ps2 = spearmanr(rr, ff)
    print(f"\nexternal corpus, merging methods only ({len(merging)}):")
    print(f"  recovery vs macro-F1   r = {r2:+.3f}  p = {p2:.4f}   rho = {s2:+.3f}  p = {ps2:.4f}")
    print(f"  main corpus            r = -0.798   p = 0.0057    rho = -0.857  p = 0.0015")

print("\nconstraint pairs on the external corpus:")
for u, g in [("soundex","soundex_guard"), ("indicsoundex","indicsoundex_guard"),
             ("lccn_base","lccn")]:
    if u in ext_mean and g in ext_mean:
        n_pos = int(sum(1 for _, row in rep.iterrows() if row[g] > row[u]))
        print(f"  {u:<20}{ext_mean[u]:.4f} -> {g:<20}{ext_mean[g]:.4f}"
              f"   {ext_mean[g]-ext_mean[u]:+.4f}   positive in {n_pos}/{len(rep)} models")

worst = min(shared, key=lambda a: ext_mean[a])
print(f"\nworst method on the external corpus: {worst} ({ext_mean[worst]:.4f}, "
      f"{ext_mean[worst]-ext_mean['raw']:+.4f} vs raw)")
print(f"worst on the main corpus:           indicsoundex (0.7530, -0.0267 vs raw)")


methods compared: 12
Spearman rank correlation between corpora: rho = +0.634  p = 0.0268
Pearson: r = +0.890  p = 0.0001

            method  main (9 models)  external (4 models)  main rank  ext rank  shift
              kope           0.7807               0.4516          1         6      5
               raw           0.7797               0.4556          2         5      3
       lccn_itrans           0.7795               0.4496          3         7      4
         lccn_base           0.7783               0.4652          4         1     -3
              keel           0.7776               0.4577          5         3     -2
              lccn           0.7776               0.4603          5         2     -3
             terun           0.7739               0.4572          7         4     -3
         metaphone           0.7649               0.4450          8         8      0
indicsoundex_guard           0.7594               0.4241          9        12      3
     soundex_guard          

In [6]:
# ================================================================
#  CELL 5 — Q2 TRANSFER, and export
#  Train on our corpus, test on the external one. Absolute scores will fall
#  well below the within-corpus numbers because of the platform shift; the
#  question is whether the ORDERING of methods survives it.
# ================================================================
sp = glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)
tra = None
if not sp:
    print("no NB1 splits attached; skipping the transfer experiment")
else:
    mtr = pd.read_csv(os.path.dirname(sp[0]) + "/train.csv",
                      dtype=str, keep_default_na=False)
    mtr.columns = [c.strip().lower() for c in mtr.columns]
    LAB = next(c for c in ("majority", "gold_label", "label") if c in mtr.columns)
    mtr = mtr[mtr[LAB].isin(LABELS)]
    print(f"main train {len(mtr):,} -> external test {len(ete):,}")

    rows = []
    for name, ctor in MODELS.items():
        r = {"model": name}
        for a in ARMS:
            col = f"text_{a}"
            if col not in mtr.columns: continue
            v = vec()
            Xtr = v.fit_transform(mtr[col]); Xte = v.transform(ete[col])
            r[a] = round(f1_score(ete.label,
                                  ctor().fit(Xtr, mtr[LAB]).predict(Xte),
                                  average="macro"), 4)
        rows.append(r)
    tra = pd.DataFrame(rows)
    tra.to_csv("/kaggle/working/cross_transfer.csv", index=False)
    print("\n=== Q2 TRANSFER, macro-F1 ===")
    print(tra.to_string(index=False))

    cols = [a for a in ARMS if a in tra.columns]
    tm = {a: tra[a].mean() for a in cols}
    sh2 = [a for a in cols if a in MAIN]
    if len(sh2) >= 4:
        rho2, p3 = spearmanr([MAIN[a] for a in sh2], [tm[a] for a in sh2])
        print(f"\nrank correlation with the main corpus: rho = {rho2:+.3f}  p = {p3:.4f}")
    print(f"\nmean across methods: within-corpus {np.mean(list(ext_mean.values())):.4f}"
          f"  transfer {np.mean(list(tm.values())):.4f}")
    print("The drop is the platform shift; the ordering is the result.")

print("\n" + "="*66)
print("SUMMARY FOR THE PAPER")
print("="*66)
print(f"external corpus       {os.path.basename(EXT_PATH)}, {len(ext):,} comments")
print(f"                      {len(etr):,} train / {len(ete):,} test")
print(f"methods               {len(ARMS)}")
print(f"models                {len(MODELS)}")
print(f"rank correlation      rho = {rho:+.3f}  p = {p_rho:.4f}")
print(f"pipeline              {FP}")
print("\ntables written to /kaggle/working:")
for f in ("cross_change_rates", "cross_replication", "cross_ranking", "cross_transfer"):
    p = f"/kaggle/working/{f}.csv"
    print("  ", f + ".csv" if os.path.exists(p) else f + ".csv  (not produced)")


main train 28,000 -> external test 1,099

=== Q2 TRANSFER, macro-F1 ===
              model    raw   keel   kope  metaphone  terun  soundex  soundex_guard  indicsoundex  indicsoundex_guard  lccn_base   lccn  lccn_itrans
Logistic Regression 0.4046 0.4060 0.4146     0.3773 0.4059   0.3078         0.3321        0.3185              0.3347     0.4173 0.3980       0.3902
                SVM 0.3990 0.3977 0.4189     0.3682 0.4031   0.3188         0.3342        0.3288              0.3448     0.4202 0.4093       0.4022
        Naive Bayes 0.2818 0.2888 0.2875     0.2599 0.2712   0.2349         0.2336        0.2325              0.2421     0.2883 0.2814       0.3130
      Random Forest 0.3013 0.2994 0.2991     0.2955 0.3072   0.2437         0.2474        0.2661              0.2418     0.2918 0.3027       0.3084

rank correlation with the main corpus: rho = +0.879  p = 0.0002

mean across methods: within-corpus 0.4465  transfer 0.3265
The drop is the platform shift; the ordering is the result.

SU